# Weekly Challenge 03 – Help-Desk Router 💻
*Text classification · difficulty ★★★☆ · ~1 evening*

A 1990s university help desk receives messages about computers and has to forward each one to the right
expert. There are **five queues**, and they overlap a lot:

`comp.graphics` · `comp.os.ms-windows.misc` · `comp.sys.ibm.pc.hardware` · `comp.sys.mac.hardware` · `comp.windows.x`

You get the raw message text (headers, signatures and quoted replies removed, so you can't cheat by reading the newsgroup name).
The current router – word counts + Naive Bayes – gets a macro-F1 of only about 0.55. Your job: **reach macro-F1 ≥ 0.70.**

| | |
|---|---|
| **Data** | `sklearn.datasets.fetch_20newsgroups`, the 5 categories above, `remove=("headers", "footers", "quotes")` (downloads ≈ 14 MB once, then cached) |
| **Split** | the official `train` (2 936 docs) / `test` (1 955 docs) split by date |
| **Metric** | macro-averaged F1 on the test set |
| **Baseline** | `CountVectorizer` + `MultinomialNB` → **0.545** |
| **Target** | **≥ 0.70** |

Theory: [[Text Representations]] · [[NLP Overview]] · [[Naive Bayes]] · [[Support Vector Machines]] · [[Model Evaluation and Metrics]]

**How it works**
1. Read the story, run the setup and the baseline.
2. Build your model in the ✏️ *Your turn* cell (it starts as a copy of the baseline).
3. Run the 🏁 *Evaluate* cell – it prints ✅ if you beat the target, ❌ otherwise.
4. Stuck? Open the 💡 hints one at a time. Done (or truly stuck)? Open the 🏆 solution walkthrough.
5. Log your best score in [[Weekly Challenges]].

**Rules** – make every modelling decision with the training data only (cross-validation or a validation split).
The test set is touched exactly once, by the evaluate cell. Peeking at it to tune is cheating yourself.

In [ ]:
import numpy as np, pandas as pd, warnings
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report
warnings.filterwarnings("ignore")

def report(name, score, baseline, target, higher_is_better=True):
    beat = score >= target if higher_is_better else score <= target
    arrow = "≥" if higher_is_better else "≤"
    print(f"{name}: {score:.4f}   (baseline {baseline:.4f}, target {arrow} {target})")
    print(("✅ Target beaten – log it in Weekly Challenges!" if beat
           else "❌ Not yet – keep going (or open a hint)."))

## 1 · Load the data (fixed split)

In [ ]:
CATS = ["comp.graphics", "comp.os.ms-windows.misc", "comp.sys.ibm.pc.hardware", "comp.sys.mac.hardware", "comp.windows.x"]
kw = dict(categories=CATS, remove=("headers", "footers", "quotes"))
train, test = fetch_20newsgroups(subset="train", **kw), fetch_20newsgroups(subset="test", **kw)
X_train, y_train, X_test, y_test = train.data, train.target, test.data, test.target
names = train.target_names
print(len(X_train), "train docs,", len(X_test), "test docs")

## 2 · Have a look
Read a few messages. How long are they? How many are (almost) empty?

In [ ]:
lengths = pd.Series([len(t.split()) for t in X_train])
print(lengths.describe().round(1).to_dict())
print("empty or < 5 words:", int((lengths < 5).sum()))
for i in range(2):
    print(f"\n--- {names[y_train[i]]} ---\n{X_train[i][:400]}")

## 3 · Baseline

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

baseline = make_pipeline(CountVectorizer(), MultinomialNB()).fit(X_train, y_train)
BASELINE = f1_score(y_test, baseline.predict(X_test), average="macro")
TARGET = 0.70
print(f"baseline test macro-F1: {BASELINE:.3f}")

## 4 · ✏️ Your turn
Define `my_model` – a pipeline that takes **raw strings** and predicts a class. Compare ideas with cross-validated macro-F1.

In [ ]:
# ✏️ TODO: replace this with your own model
my_model = make_pipeline(CountVectorizer(), MultinomialNB())

cv = cross_val_score(my_model, X_train, y_train, cv=5, scoring="f1_macro")
print(f"5-fold CV macro-F1 on train: {cv.mean():.3f} ± {cv.std():.3f}")

## 5 · 🏁 Evaluate

In [ ]:
my_model.fit(X_train, y_train)
pred = my_model.predict(X_test)
score = f1_score(y_test, pred, average="macro")
report("test macro-F1", score, BASELINE, TARGET)
print(classification_report(y_test, pred, target_names=names, digits=3))

<details><summary>💡 Hint 1</summary>

Raw counts let long messages and very common words ("the", "is", "it") dominate. How could you down-weight words that appear everywhere?
</details>

<details><summary>💡 Hint 2</summary>

Swap `CountVectorizer` for `TfidfVectorizer`. Options worth trying: `sublinear_tf=True`, `stop_words="english"`, `ngram_range=(1, 2)`, `min_df=2`.
</details>

<details><summary>💡 Hint 3</summary>

With TF-IDF features, a linear classifier that maximises the margin (`LinearSVC`) or `SGDClassifier` usually beats Naive Bayes. Also try `MultinomialNB(alpha=...)` with a much smaller alpha.
</details>

<details>
<summary>🏆 <b>Solution walkthrough</b> (click to open – try first!)</summary>

**Why the baseline is weak.** With raw counts, Naive Bayes multiplies one probability per word *occurrence*. Long
posts full of generic words ("the", "drive", "problem") swamp the few words that actually identify the queue, and the
default smoothing `alpha=1` is far too strong for a 53 000-word vocabulary. The result: it almost never predicts
`comp.os.ms-windows.misc` (recall 0.003 – look at the baseline's classification report).

**Step 1 – TF-IDF.** `TfidfVectorizer(sublinear_tf=True, stop_words="english")` uses log term frequencies, down-weights
words that occur in many documents and normalises every document to unit length. Even keeping Naive Bayes
(with `alpha=0.01`) this jumps to ≈ 0.69.

**Step 2 – a discriminative linear model.** `LinearSVC` on the same features learns weights directly for the
decision, instead of modelling each word independently. It reaches **≈ 0.726 test macro-F1**. Bigrams +
`SGDClassifier` lands in the same place (≈ 0.724).

**Why not 0.9?** The five queues genuinely overlap (a post about a Windows graphics driver belongs to two of them), and
after removing headers 98 training posts have fewer than five words. In the classification report
`comp.os.ms-windows.misc` is still the hardest queue (F1 ≈ 0.67), and IBM-PC vs. Mac hardware get mixed up 93 times.

**Lesson.** For text classification, TF-IDF + a linear model is the baseline every fancy model has to beat.

```python
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

my_model = make_pipeline(TfidfVectorizer(sublinear_tf=True, stop_words="english"), LinearSVC())
cv = cross_val_score(my_model, X_train, y_train, cv=5, scoring="f1_macro")
print(f"5-fold CV macro-F1 on train: {cv.mean():.3f} ± {cv.std():.3f}")
```
</details>